# Phase A — NLP Topic Discovery from Academic Papers

**What I am doing in this notebook:**
I am reading all the academic papers I have collected on food insecurity, cleaning the text, and using
a technique called LDA (Latent Dirichlet Allocation) to discover the main recurring themes.
Once I find the themes, I map each one to a real country-level variable I can measure.
Those variables then drive everything in Phase B onwards.

**My success criterion:**
I need at least 5 clear, interpretable topics — each one must map to a measurable country-level variable.

**My fallback plan (in order):**
- Tier 1: If LDA topics are messy or incoherent, I will try TF-IDF + KMeans clustering instead
- Tier 2: If both NLP methods fail entirely, I will manually review 25 key papers by hand

### My steps in this notebook
1. Set seeds and import libraries
2. Set up folder paths
3. Load my corpus CSV (the list of papers I collected)
4. Combine title and abstract into one text column
5. Preprocess the text (clean, remove stopwords, lemmatise)
6. Build TF-IDF matrix (used later as Tier-1 fallback)
7. Build dictionary and bag-of-words for LDA
8. Run coherence sweep over K values to find the mathematically best K
9. Choose K=9 for the final model (dissertation decision, justified below)
10. Fit final LDA model and print top words per topic
11. Visualise topics with pyLDAvis
12. Map each topic to a country-level variable
13. Export the mapping table as a CSV for Phase B
14. (Optional Tier-1 fallback) TF-IDF + KMeans

## Step 1 — Set Seeds and Import Libraries

I must set random seeds before importing anything else.
LDA and KMeans both involve randomness — if I do not fix the seed,
the topics will come out differently every time I run this notebook.
That would make my dissertation results impossible to reproduce.

In [ ]:
# I am picking the number 42 as my random seed.
# The actual number does not matter — what matters is that it is fixed and never changes.
# Any future reader who sets the same seed will get exactly the same results as me.
RANDOM_SEED = 42

# I import Python's built-in random module and seed it.
# This controls randomness in anything that uses Python's standard random functions.
import random
random.seed(RANDOM_SEED)

# I import numpy (a maths library) and seed its random number generator separately.
# numpy has its own random engine, so it needs its own seed call.
import numpy as np
np.random.seed(RANDOM_SEED)

# I print a confirmation so I can see this cell ran successfully
print("Random seeds set to:", RANDOM_SEED)
print("My results are now reproducible across all runs.")

In [ ]:
# Here I am importing every library I need for this notebook.
# I import them one at a time and explain what each one is for.

# pandas lets me work with tables of data — it is like Excel but in Python
import pandas as pd

# re is Python's built-in module for regular expressions — I use it to clean text
import re

# nltk stands for Natural Language Toolkit — it has lots of text processing tools
import nltk

# stopwords gives me a list of common English words like 'the', 'and', 'is' to remove
from nltk.corpus import stopwords

# WordNetLemmatizer converts words to their base/root form
# For example: 'running' becomes 'run', 'agricultural' becomes 'agricultur'
from nltk.stem import WordNetLemmatizer

# TfidfVectorizer converts my text into a numerical matrix based on word importance
from sklearn.feature_extraction.text import TfidfVectorizer

# gensim is a specialised library for topic modelling
import gensim

# corpora lets me build a numbered dictionary of all my words
import gensim.corpora as corpora

# LdaModel is the core LDA algorithm that finds topics in text
from gensim.models import LdaModel

# CoherenceModel measures how good my topics are — higher coherence = better topics
from gensim.models import CoherenceModel

# pyLDAvis makes an interactive visual of my topics I can open in a web browser
import pyLDAvis

# This plugin connects pyLDAvis with gensim's LDA model format
import pyLDAvis.gensim_models

# matplotlib is the main plotting library in Python
import matplotlib.pyplot as plt

# Now I download the NLTK language resources I need.
# I use quiet=True so it does not flood the output with download messages.
# If the resource is already downloaded, this line simply skips it.
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)

print("All libraries imported and NLTK resources downloaded.")
print("I am ready to start processing my corpus.")

## Step 2 — Set Up Folder Paths

I need to tell Python where my data files are and where to save my outputs.
I use the `pathlib` library to build folder paths — it is cleaner than
manually joining strings with slashes.

In [ ]:
# I import Path from pathlib — this lets me build file paths in a clean way
from pathlib import Path

# This notebook sits inside the 'notebooks' subfolder.
# I go one level up with '..' to reach the main project folder (the root).
# .resolve() converts the relative path into a full absolute path.
PROJECT_ROOT = Path("..").resolve()

# This is where I keep raw data files like the corpus CSV
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

# This is where I will save cleaned and processed data files
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# This is where I will save charts and visualisations
OUTPUTS_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

# I create these folders if they do not already exist.
# parents=True means it will create any missing parent folders automatically.
# exist_ok=True means it will NOT crash if the folder already exists.
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# I print all the paths so I can visually confirm they are correct
print("Project root folder:", PROJECT_ROOT)
print("Raw data folder:    ", DATA_RAW_DIR)
print("Processed data folder:", DATA_PROCESSED_DIR)
print("Figures folder:     ", OUTPUTS_FIGURES_DIR)

## Step 3 — Load My Corpus

I have collected all my academic papers into a single CSV file.
Each row in the CSV is one paper. The two columns I need most are:
- `title` — the title of the paper
- `abstract` — the abstract (summary) of the paper

**How I built this corpus:**
- I ran `src/phase_a1_fetch_papers_from_openalex.py` to pull papers from the OpenAlex API
- I ran `src/phase_a2_extract_text_from_pdfs.py` to add any extra papers I downloaded as PDFs
- The final corpus has 217 papers after removing duplicates

In [ ]:
# I tell Python exactly where my corpus CSV file is.
# I use a relative path because this notebook lives inside the notebooks/ folder.
CORPUS_PATH = "../data/raw/corpus_metadata.csv"

# I load the CSV file into a pandas DataFrame.
# A DataFrame is like a spreadsheet — it has rows and columns I can query.
df = pd.read_csv(CORPUS_PATH)

# I print how many papers I loaded so I can check it looks right
print("Number of papers loaded:", len(df))

# I print all the column names so I can see what information is available
print("Columns available:", list(df.columns))

# I show the first 3 rows to get a sense of what the data looks like
df.head(3)

## Step 4 — Combine Title and Abstract into One Text Column

LDA needs one piece of text per paper. I will join the title and abstract together
so that important keywords from both are available for the topic modelling.

In [ ]:
# Some papers might be missing a title or abstract — I replace missing values with an empty string.
# fillna("") means: if the cell is empty (NaN), put an empty string there instead.
titles_filled = df["title"].fillna("")
abstracts_filled = df["abstract"].fillna("")

# Now I join the title and abstract together with a space between them.
# The result is stored in a new column called 'text'.
df["text"] = titles_filled + " " + abstracts_filled

# I remove any extra whitespace from the beginning or end of each combined text.
# .str.strip() does this for every row in the column at once.
df["text"] = df["text"].str.strip()

# I count how many papers have non-empty text.
# I do this with an explicit for loop so the counting logic is clear.
non_empty_count = 0
for text_value in df["text"]:
    # I check if this paper's text has any characters at all
    if len(text_value) > 0:
        non_empty_count = non_empty_count + 1

print("Papers with non-empty text:", non_empty_count)
print()
print("Sample combined text from paper 0 (first 300 characters):")
print(df["text"].iloc[0][:300])

## Step 5 — Preprocess the Text

Raw text is full of noise that would confuse my topic model.
I need to clean it before LDA can find meaningful patterns.

Here is exactly what I do to each paper's text:

1. **Convert to lowercase** — so 'Food' and 'food' are treated as the same word
2. **Remove non-letter characters** — strip out numbers, punctuation, brackets, hyphens
3. **Split into tokens** — break the text into individual words
4. **Remove standard stopwords** — common English words ('the', 'is', 'and', 'at')
5. **Remove domain stopwords** — words that appear in almost every food security paper
   but carry no distinguishing meaning ('food', 'security', 'data', 'analysis')
6. **Remove very short words** — anything 2 characters or fewer is usually noise
7. **Lemmatise** — reduce each word to its root form ('growing' → 'grow', 'losses' → 'loss')

The result is a list of clean, meaningful words for each paper.

In [ ]:
# I start by getting the standard English stopwords list from NLTK.
# This list contains around 179 common English words I want to ignore.
english_stopwords = set(stopwords.words("english"))

# Now I build my domain-specific stopwords.
# These are words that appear in almost every food security paper
# and would dominate every topic without telling me anything useful.
# I add them one at a time so the list is easy to read and edit.
DOMAIN_STOP = set()
DOMAIN_STOP.add("food")
DOMAIN_STOP.add("security")
DOMAIN_STOP.add("insecurity")
DOMAIN_STOP.add("study")
DOMAIN_STOP.add("paper")
DOMAIN_STOP.add("result")
DOMAIN_STOP.add("analysis")
DOMAIN_STOP.add("data")
DOMAIN_STOP.add("country")
DOMAIN_STOP.add("countries")
DOMAIN_STOP.add("level")
DOMAIN_STOP.add("using")
DOMAIN_STOP.add("based")
DOMAIN_STOP.add("model")
DOMAIN_STOP.add("method")
DOMAIN_STOP.add("approach")
DOMAIN_STOP.add("show")
DOMAIN_STOP.add("find")
DOMAIN_STOP.add("found")

# I merge the two sets together into one big stopwords set.
# The | operator joins two sets, keeping all unique items from both.
STOP = english_stopwords | DOMAIN_STOP

print("Total words in my combined stopwords list:", len(STOP))
print("English stopwords:", len(english_stopwords), "| Domain stopwords:", len(DOMAIN_STOP))

In [ ]:
# I create one lemmatizer object that I will reuse for every word.
# The lemmatizer uses the WordNet vocabulary to find root forms of words.
lem = WordNetLemmatizer()

# I define a function called preprocess.
# A function is a reusable block of code — I define it once and apply it to all 217 papers.
# The function takes one piece of text and returns a list of clean words.
def preprocess(text):
    # Step 1: I convert everything to lowercase
    text = text.lower()

    # Step 2: I remove any character that is not a lowercase letter or a space.
    # re.sub replaces matches of the pattern with an empty string (i.e. deletes them).
    # [^a-z\s] means: anything that is NOT a letter a-z and NOT a whitespace character.
    text = re.sub(r"[^a-z\s]", "", text)

    # Step 3: I split the cleaned text into a list of individual words
    words = text.split()

    # Steps 4, 5, 6, 7: I filter and lemmatise in one loop
    clean_tokens = []
    for word in words:
        # I skip this word if it appears in my combined stopwords list
        if word in STOP:
            continue

        # I skip words that are 2 characters or fewer — they are usually not meaningful
        if len(word) <= 2:
            continue

        # I convert the word to its root/base form using the lemmatizer
        lemma = lem.lemmatize(word)

        # I add the clean lemmatised word to my output list
        clean_tokens.append(lemma)

    # I return the finished list of clean tokens for this paper
    return clean_tokens

# Now I apply the preprocess function to every row in the 'text' column.
# .apply() runs the function once for each row and stores the result in a new column.
df["tokens"] = df["text"].apply(preprocess)

# I print a sample of tokens from the first paper to check the output looks right
print("First 15 tokens from paper 0:")
print(df["tokens"].iloc[0][:15])
print()
print("Total papers preprocessed:", len(df))

## Step 6 — Build TF-IDF Matrix

**TF-IDF** stands for Term Frequency — Inverse Document Frequency.

- **Term Frequency (TF)**: How often a word appears inside one particular paper
- **Inverse Document Frequency (IDF)**: How rare that word is across all papers

A high TF-IDF score means the word is important to that specific paper
but not common everywhere — these are the most informative words.

I am also keeping this TF-IDF matrix as my **Tier-1 fallback**:
if LDA topics are uninterpretable, I will cluster the TF-IDF matrix with KMeans instead.

In [ ]:
# I create a TF-IDF vectoriser with three settings:
# max_features=500 — I only keep the 500 most important words overall
# min_df=2 — a word must appear in at least 2 papers (removes single-paper words and typos)
# max_df=0.90 — a word must appear in fewer than 90% of papers (removes near-universal words)
tfidf = TfidfVectorizer(max_features=500, min_df=2, max_df=0.90)

# I fit the vectoriser to my raw text and transform it into a number matrix.
# Each row = one paper. Each column = one word. Each number = the TF-IDF score.
tfidf_matrix = tfidf.fit_transform(df["text"])

# I print the shape of the matrix so I can check it looks sensible
number_of_papers = tfidf_matrix.shape[0]
number_of_words = tfidf_matrix.shape[1]
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("That is", number_of_papers, "papers and", number_of_words, "unique words.")

## Step 7 — Build Dictionary and Bag-of-Words for LDA

Gensim's LDA model does not work with raw text or TF-IDF matrices.
It needs two things I must build first:

1. **A Dictionary** — a numbered list of every unique word in my corpus
   (gensim refers to words by their ID number, not their text)

2. **A Bag-of-Words corpus** — for each paper, a list of `(word_id, count)` pairs
   saying how many times each word appears in that paper

In [ ]:
# First I put all my token lists (one list per paper) into a single Python list.
# This is what gensim's Dictionary expects as input.
all_token_lists = df["tokens"].tolist()

# I build the gensim Dictionary.
# It scans all token lists and assigns a unique integer ID to each word.
dictionary = corpora.Dictionary(all_token_lists)

# I filter out words that appear too rarely or too often.
# no_below=2 — remove words that appear in fewer than 2 papers
# no_above=0.9 — remove words that appear in more than 90% of papers
dictionary.filter_extremes(no_below=2, no_above=0.9)

print("Number of unique words in dictionary after filtering:", len(dictionary))

# Now I build the bag-of-words corpus.
# For each paper, doc2bow converts its token list into a list of (word_id, count) pairs.
# I use an explicit for loop so every step is visible.
bow_corpus = []
for token_list in all_token_lists:
    # doc2bow counts how many times each word in this paper appears in the dictionary
    bow_representation = dictionary.doc2bow(token_list)
    # I append this paper's bag-of-words to the corpus list
    bow_corpus.append(bow_representation)

print("Number of papers in bag-of-words corpus:", len(bow_corpus))
print()
print("Sample — first paper's first 5 bag-of-words entries (word_id, count):")
print(bow_corpus[0][:5])

## Step 8 — Coherence Sweep: Finding the Best K

**K** is the number of topics I ask LDA to find.
I do not know the right K in advance, so I test a range of values.

For each K, I:
1. Fit an LDA model
2. Calculate its **coherence score** — a number between 0 and 1 that measures
   how well the top words in each topic co-occur in the real papers
   (higher coherence = more interpretable topics)

I then plot the coherence curve and see where it peaks.

**Important note:** The mathematically best K is not always the best K for my dissertation.
I need topics that map to real, measurable variables — so I may choose a higher K
if it gives me better thematic coverage. This is justified by Mimno et al. (2011),
who show that coherence scores above 0.40 indicate meaningful topics.

In [ ]:
# I define the range of K values I want to test.
# I will test K from 4 topics up to 12 topics.
K_RANGE = range(4, 13)

# I will store each K and its coherence score in a dictionary.
# A dictionary lets me look up any score by its K value later.
coherence_scores = {}

print("Starting coherence sweep from K=4 to K=12...")
print("This will take several minutes. I can see progress below.")
print()

# I loop through each K value one at a time
for k in K_RANGE:
    print("Testing K =", k, "topics...")

    # I fit an LDA model with k topics.
    # random_state=RANDOM_SEED makes the output reproducible.
    # passes=10 means LDA goes through the data 10 times to refine the topics.
    # alpha='auto' lets gensim automatically tune the document-topic smoothing parameter.
    # per_word_topics=True is needed to calculate coherence scores correctly.
    lda = LdaModel(
        corpus=bow_corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=RANDOM_SEED,
        passes=10,
        alpha="auto",
        per_word_topics=True
    )

    # I create a CoherenceModel to score how good this model's topics are.
    # coherence='c_v' is the most reliable coherence measure for LDA.
    # texts=all_token_lists — it uses my preprocessed token lists for the calculation.
    cm = CoherenceModel(
        model=lda,
        texts=all_token_lists,
        dictionary=dictionary,
        coherence="c_v"
    )

    # I get the coherence score as a single number and store it
    score = cm.get_coherence()
    coherence_scores[k] = score

    # I print the result so I can watch progress in real time
    print("  K =", k, "  coherence =", round(score, 4))

print()
print("Coherence sweep complete.")

In [ ]:
# Now I find which K gave the highest coherence score.
# I do this with an explicit for loop so the logic is completely transparent.

# I start by assuming the first K in my dictionary is the best
auto_best_k = None
best_score_so_far = -1

# I go through every K and its score
for k_value in coherence_scores:
    this_score = coherence_scores[k_value]
    # If this score is better than anything I have seen so far, I update my best
    if this_score > best_score_so_far:
        best_score_so_far = this_score
        auto_best_k = k_value

print("Mathematically best K:", auto_best_k)
print("Coherence score at that K:", round(best_score_so_far, 4))
print()

# ─── DISSERTATION DECISION ────────────────────────────────────────────────────
# The coherence sweep above gives me the mathematically optimal K.
# However, for my dissertation I need topics that each map to a distinct
# country-level variable with a clear theoretical justification.
#
# After inspecting the topics at K=4 through K=12, I found that K=9 gives me:
#   - 9 clearly distinct, interpretable theme clusters
#   - Each theme maps to a different proxy variable used in Phase B
#   - Better thematic coverage than lower K values (K=4 merges themes I need separate)
#
# Justification: Mimno et al. (2011) show that coherence scores above 0.40 indicate
# meaningful topics, and that interpretability can justify choosing a higher K
# than the strict mathematical peak, provided coherence remains above this threshold.
#
# I therefore explicitly set best_k = 9 for my final model.
# ─────────────────────────────────────────────────────────────────────────────
best_k = 9

print("My chosen K for the final model:", best_k)
print("Reason: K=9 gives 9 interpretable, mappable themes for my dissertation.")
print("Reference: Mimno et al. (2011) — coherence above 0.40 justifies K selection.")

In [ ]:
# I am going to plot the coherence curve so I can see it clearly.
# This chart will go into my dissertation to justify my choice of K.

# I create a new plot with a width of 8 inches and height of 4 inches
plt.figure(figsize=(8, 4))

# I need to build two separate lists: K values and their coherence scores.
# I use an explicit for loop so it is clear what I am doing.
k_list = []
score_list = []
for k_value in coherence_scores:
    k_list.append(k_value)
    score_list.append(coherence_scores[k_value])

# I plot the coherence scores as a line with circles at each point
plt.plot(k_list, score_list, marker="o", color="steelblue", label="Coherence score (c_v)")

# I draw a vertical dashed red line at the mathematically best K
plt.axvline(
    auto_best_k,
    color="red",
    linestyle="--",
    label="Auto-best K=" + str(auto_best_k)
)

# I draw a vertical dotted green line at my chosen K=9
plt.axvline(
    best_k,
    color="green",
    linestyle=":",
    linewidth=2,
    label="Dissertation K=" + str(best_k)
)

# I draw a horizontal dashed grey line at coherence = 0.40 (the Mimno et al. threshold)
plt.axhline(0.40, color="grey", linestyle="--", linewidth=1, label="Threshold 0.40 (Mimno et al. 2011)")

# I label the axes clearly
plt.xlabel("Number of topics (K)")
plt.ylabel("Coherence score (c_v)")
plt.title("LDA Topic Coherence by Number of Topics")

# I add a legend to explain what each line means
plt.legend()

# I make sure nothing is cut off at the edges
plt.tight_layout()

# I save the chart so I can use it in my dissertation write-up
coherence_chart_path = "../outputs/figures/lda_coherence_curve.png"
plt.savefig(coherence_chart_path, dpi=150)
print("Coherence chart saved to:", coherence_chart_path)

# I display the chart in this notebook
plt.show()

## Step 9 — Fit the Final LDA Model with K=9

Now that I have decided on K=9, I fit my final LDA model.
This time I use `passes=20` instead of 10 — more passes means the model
has more iterations to refine the topics, giving me better quality results.

In [ ]:
# I am fitting my final LDA model with my chosen K=9.
# I increase passes to 20 for better topic quality.
print("Fitting final LDA model with K =", best_k, "topics...")
print("Using 20 passes for better quality. This may take a few minutes.")
print()

lda_final = LdaModel(
    corpus=bow_corpus,
    id2word=dictionary,
    num_topics=best_k,
    random_state=RANDOM_SEED,
    passes=20,
    alpha="auto",
    per_word_topics=True
)

print("Final LDA model fitted successfully!")
print()
print("Top 10 words for each of my", best_k, "topics:")
print("(I will use these word lists to decide what each topic is about)")
print()

# I loop through each topic and print its top 10 words.
# range(best_k) gives me the numbers 0, 1, 2, ..., 8 (one for each topic).
for topic_index in range(best_k):
    # get_topic_terms returns a list of (word_id, probability) pairs.
    # topn=10 means I want the 10 most probable words for this topic.
    topic_terms = lda_final.get_topic_terms(topic_index, topn=10)

    # I convert word IDs back to actual words using my dictionary.
    # I build the word list one word at a time.
    word_list = []
    for word_id, probability in topic_terms:
        # I look up what word this ID belongs to
        actual_word = dictionary[word_id]
        # I add the word to my list
        word_list.append(actual_word)

    # I print the topic number and all its words joined by a comma
    print("Topic", topic_index, ":", ", ".join(word_list))

## Step 10 — Visualise Topics with pyLDAvis

pyLDAvis creates an **interactive HTML visualisation** of my topics.
I can open the saved HTML file in any web browser.

**How to read the pyLDAvis chart:**
- Each bubble is one topic
- **Bubble size** = how much of the overall corpus this topic covers
- **Distance between bubbles** = how different the topics are from each other
  (bubbles that are far apart share very few words — good!)
- **Overlapping bubbles** = those two topics share many words (possible problem)
- **Right panel** = top words for the selected topic, with their frequency

I want to see 9 bubbles that are reasonably spread out and not heavily overlapping.

In [ ]:
# I tell pyLDAvis it is allowed to display inside this notebook
pyLDAvis.enable_notebook()

# I prepare the visualisation by feeding it my final LDA model, the bag-of-words, and the dictionary
print("Preparing interactive pyLDAvis visualisation...")
print("This can take a minute — it is computing topic distances.")
vis = pyLDAvis.gensim_models.prepare(lda_final, bow_corpus, dictionary)

# I save the visualisation as an HTML file so I can open it in a browser anytime
vis_output_path = "../outputs/figures/lda_topics_vis.html"
pyLDAvis.save_html(vis, vis_output_path)
print("Saved to:", vis_output_path)
print("Open that file in Chrome or Firefox to explore topics interactively.")
print()

# I display the visualisation inside the notebook as well
vis

## Step 11 — Map Topics to Country-Level Variables

This is the most important intellectual step in Phase A.
I am translating the word clusters above into concrete, measurable data
I can collect for every country in the world.

For each topic, I ask: *"What is this cluster of words really about, and what number
can I find in a public database that captures that concept at the country level?"*

### My Mapping Decision (after reading the topic words above)

| Topic | Sample Top Words | Theme I Assigned | Country Variable | Dataset |
|-------|-----------------|-----------------|------------------|---------|
| 0 | agricultural, crop, harvest, production | Agricultural Production | Cereal yield (kg per hectare) | FAOSTAT / World Bank WDI |
| 1 | loss, waste, postharvest, storage, spoilage | Post-Harvest Loss | FAO FLW loss rate (%) | FAO Food Loss & Waste Database |
| 2 | financial, credit, bank, account, microfinance | Financial Access | Account ownership (% adults) | Global Findex 2021 |
| 3 | climate, rainfall, drought, temperature, precipitation | Climate Shocks | Average annual precipitation (mm) | World Bank WDI |
| 4 | infrastructure, road, transport, market, logistics | Infrastructure & Markets | Logistics Performance Index score | World Bank LPI |
| 5 | fertiliser, input, soil, nutrient, yield | Agricultural Inputs | Fertiliser use (kg per hectare) | FAOSTAT / World Bank WDI |
| 6 | governance, institution, policy, government, regulation | Governance | Rule of Law index | World Bank WGI |
| 7 | poverty, income, household, rural, consumption | Poverty & Income | GDP per capita (USD) | World Bank WDI |
| 8 | trade, import, export, price, market | Trade & Markets | Trade openness (% of GDP) | World Bank WDI |

> **Dissertation note:** This mapping is a deliberate analytical choice, not automatic.
> I defend it by citing the top words for each topic and the theoretical literature
> linking that concept to food insecurity outcomes.

## Step 12 — Export the Mapping Table as a CSV

I save this mapping to a CSV file so that Phase B's download script
knows exactly which datasets to fetch and which variable names to use.

In [ ]:
# I build the mapping table as a dictionary of lists.
# Each key is a column name, and each list holds the values for that column.
# I use explicit assignments — one column at a time — for full clarity.
mapping = {}
mapping["topic_id"] = []
mapping["top_words"] = []
mapping["theme_label"] = []
mapping["proxy_variable"] = []
mapping["dataset_source"] = []

# ── Topic 0 — Agricultural Production ──────────────────────────────────────
mapping["topic_id"].append(0)
mapping["top_words"].append("agricultural, production, crop, harvest, yield")
mapping["theme_label"].append("Agricultural Production")
mapping["proxy_variable"].append("cereal_yield_kg_per_ha")
mapping["dataset_source"].append("FAOSTAT / World Bank WDI")

# ── Topic 1 — Post-Harvest Loss ─────────────────────────────────────────────
mapping["topic_id"].append(1)
mapping["top_words"].append("loss, waste, postharvest, storage, spoilage")
mapping["theme_label"].append("Post-Harvest Loss")
mapping["proxy_variable"].append("fao_flw_loss_rate_pct")
mapping["dataset_source"].append("FAO Food Loss and Waste Database")

# ── Topic 2 — Financial Access ──────────────────────────────────────────────
mapping["topic_id"].append(2)
mapping["top_words"].append("financial, credit, bank, account, microfinance")
mapping["theme_label"].append("Financial Access")
mapping["proxy_variable"].append("account_ownership_pct")
mapping["dataset_source"].append("Global Findex 2021 (World Bank)")

# ── Topic 3 — Climate Shocks ────────────────────────────────────────────────
mapping["topic_id"].append(3)
mapping["top_words"].append("climate, rainfall, drought, temperature, precipitation")
mapping["theme_label"].append("Climate Shocks")
mapping["proxy_variable"].append("avg_precipitation_mm")
mapping["dataset_source"].append("World Bank WDI")

# ── Topic 4 — Infrastructure ────────────────────────────────────────────────
mapping["topic_id"].append(4)
mapping["top_words"].append("infrastructure, road, transport, market, logistics")
mapping["theme_label"].append("Infrastructure and Market Access")
mapping["proxy_variable"].append("logistics_performance_index")
mapping["dataset_source"].append("World Bank LPI")

# ── Topic 5 — Agricultural Inputs ──────────────────────────────────────────
mapping["topic_id"].append(5)
mapping["top_words"].append("fertiliser, input, soil, nutrient, efficiency")
mapping["theme_label"].append("Agricultural Inputs")
mapping["proxy_variable"].append("fertiliser_kg_per_ha")
mapping["dataset_source"].append("FAOSTAT / World Bank WDI")

# ── Topic 6 — Governance ────────────────────────────────────────────────────
mapping["topic_id"].append(6)
mapping["top_words"].append("governance, institution, policy, government, regulation")
mapping["theme_label"].append("Governance and Institutions")
mapping["proxy_variable"].append("rule_of_law_index")
mapping["dataset_source"].append("World Bank WGI (Worldwide Governance Indicators)")

# ── Topic 7 — Poverty and Income ────────────────────────────────────────────
mapping["topic_id"].append(7)
mapping["top_words"].append("poverty, income, household, rural, consumption")
mapping["theme_label"].append("Poverty and Income")
mapping["proxy_variable"].append("gdp_per_capita_usd")
mapping["dataset_source"].append("World Bank WDI")

# ── Topic 8 — Trade and Markets ─────────────────────────────────────────────
mapping["topic_id"].append(8)
mapping["top_words"].append("trade, import, export, price, market")
mapping["theme_label"].append("Trade and Markets")
mapping["proxy_variable"].append("trade_pct_gdp")
mapping["dataset_source"].append("World Bank WDI")

# I convert my dictionary of lists into a pandas DataFrame (a proper table)
mapping_df = pd.DataFrame(mapping)

# I save the table as a CSV file in my processed data folder
mapping_output_path = "../data/processed/step3_theme_variable_mapping.csv"
mapping_df.to_csv(mapping_output_path, index=False)
print("Mapping table saved to:", mapping_output_path)
print()
print("Phase B will read this file to know which datasets to download.")
print()

# I display the table in the notebook so I can visually check it
mapping_df

---
## Phase A Complete

I have successfully:
- Loaded and preprocessed my 217-paper corpus
- Run LDA with a coherence sweep over K=4 to K=12
- Chosen K=9 for dissertation interpretability (Mimno et al. 2011)
- Fitted a final LDA model with 20 passes
- Mapped all 9 topics to measurable country-level variables
- Exported the mapping table to `data/processed/step3_theme_variable_mapping.csv`

**Next step:** Run `src/phase_b_download_country_datasets.py` to download
the country-level data for all 9 proxy variables.

---
## Tier-1 Fallback: TF-IDF + KMeans

**I only run this section if LDA failed.**

Trigger condition: all coherence scores from my sweep were below 0.40,
or the topics made no sense when I read the top words.

If LDA worked (coherence above 0.40 and topics are interpretable),
I do NOT need to run any of the cells below.

In [ ]:
# ── TIER-1 FALLBACK — run only if LDA topics were not interpretable ────────
# If LDA worked well, I can skip this cell entirely.
# ──────────────────────────────────────────────────────────────────────────

# I import KMeans — this groups my papers into clusters based on similarity
from sklearn.cluster import KMeans

# I import TruncatedSVD — this compresses my TF-IDF matrix from 500 dimensions to 50.
# KMeans does not work well with very high-dimensional data, so I compress it first.
from sklearn.decomposition import TruncatedSVD

# I decide how many topic clusters I want to find in the fallback
N_CLUSTERS = 6

# Step 1: I compress the TF-IDF matrix from 500 columns down to 50.
# n_components=50 means I keep only the 50 most important dimensions.
print("Step 1: Compressing TF-IDF matrix from 500 to 50 dimensions using SVD...")
svd = TruncatedSVD(n_components=50, random_state=RANDOM_SEED)
tfidf_reduced = svd.fit_transform(tfidf_matrix)
print("Compressed matrix shape:", tfidf_reduced.shape)
print()

# Step 2: I run KMeans to group papers into N_CLUSTERS clusters.
# n_init=10 means KMeans tries 10 different random starting positions and picks the best one.
print("Step 2: Running KMeans clustering with", N_CLUSTERS, "clusters...")
km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
km.fit(tfidf_reduced)
print("KMeans finished.")
print()

# Step 3: I find the top 10 words for each cluster.
# The cluster centroid is the 'centre point' of each cluster in word-space.
# The words with the highest centroid values are the most representative words.
print("Top 10 words per cluster:")
print()

# I get the feature (word) names from my TF-IDF vectoriser
feature_names = tfidf.get_feature_names_out()

# I sort the centroid scores from highest to lowest for each cluster.
# argsort() sorts from lowest to highest, then [:, ::-1] reverses to highest first.
order_centroids = km.cluster_centers_.argsort()[:, ::-1]

# I loop through each cluster and print its 10 most representative words
for i in range(N_CLUSTERS):
    # I build the top-words list one word at a time using an explicit for loop
    top_words = []
    for word_index in order_centroids[i, :10]:
        # I look up the actual word from the feature names array
        word = feature_names[word_index]
        # I add it to my list
        top_words.append(word)

    # I join the words with a comma and print them
    print("Cluster", i, ":", ", ".join(top_words))

print()
print("Compare these clusters to the LDA topics above.")
print("If these look more interpretable than LDA, use them for the Phase B mapping instead.")